In [38]:
if False:
    !conda env remove -n RAG-lnkedin
    !conda create -n RAG-lnkedin python=3.11 -y
    !conda activate RAG-lnkedin

    !pip install langchain langchain-core langchain-google-genai langchain-community faiss-cpu sentence-transformers tiktoken

#!pip install -U langchain-google-genai
#!pip install -U langchain langchain-core langchain-community langchain-google-genai
#!pip install -U langchain-community faiss-cpu
#!pip install -U sentence-transformers


In [ ]:
# gemini recommendation:
# Since you've fixed so many bugs today, here is the "Gold Standard" import block for a 2026 RAG project:

""" 
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter 
"""

In [5]:
# from langchain.text_splitter import CharacterTextSplitter
from langchain_text_splitters import CharacterTextSplitter
# from langchain.schema import Document
from langchain_core.documents import Document

In [6]:
import os

In [7]:
data_dir = "./Big Star Collectibles"

In [8]:
files = os.listdir(data_dir)
file_texts = []
for file in files:
    with open(f"{data_dir}/{file}") as f:
        file_text = f.read()
    text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
        chunk_size=128, chunk_overlap=32, # this is the critical line
    )
    texts = text_splitter.split_text(file_text)
    for i, chunked_text in enumerate(texts):
        file_texts.append(Document(page_content=chunked_text,metadata={
                    "doc_title": file.split(".")[0], 
                    "chunk_num": i})) 

Created a chunk of size 139, which is longer than the specified 128
Created a chunk of size 130, which is longer than the specified 128
Created a chunk of size 188, which is longer than the specified 128
Created a chunk of size 130, which is longer than the specified 128
Created a chunk of size 139, which is longer than the specified 128
Created a chunk of size 151, which is longer than the specified 128
Created a chunk of size 151, which is longer than the specified 128


In [9]:
#from langchain.vectorstores import FAISS
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

In [10]:
embeddings = HuggingFaceEmbeddings() # embed your data

/tmp/ipykernel_91675/3428753526.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings() # embed your data
/tmp/ipykernel_91675/3428753526.py:1: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings() # embed your data
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 788.59it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                    

In [11]:
# store the embedded data into a vector database
vector_store = FAISS.from_documents(
    file_texts,
    embedding=embeddings
)

In [12]:
retriever = vector_store.as_retriever()

In [13]:
from dotenv import load_dotenv

load_dotenv()

True

In [14]:
GOOGLE_API_KEY=os.getenv("GOOGLE_API_KEY")

In [15]:
# ensure the Google GenAI integration is installed in this kernel
#%pip install -q -U langchain-google-genai

#try:
	# try to import and use Google GenAI
	#

from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# except Exception:
# 	# fallback: try to use OpenAI from the package if available
# 	try:
# 		from langchain_openai import OpenAI
# 		llm = OpenAI()
# 	except Exception:
# 		# fallback: use OpenAI class already defined in the notebook, if present
# 		if "OpenAI" in globals():
# 			try:
# 				llm = globals()["OpenAI"]()
# 			except Exception:
# 				pass

# 		# final fallback: define a clear placeholder to avoid NameError later
# 		if "llm" not in globals() or not callable(globals().get("llm")):
# 			class _NoLLM:
# 				def __call__(self, *args, **kwargs):
# 					raise RuntimeError(
# 						"No working LLM available. Install/configure Google GenAI or OpenAI and re-run."
# 					)
# 			llm = _NoLLM()

#### Para OpenAI:
#from langchain_openai import ChatOpenAI
#llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY)

##### Para Anthropic:
# from langchain_anthropic import ChatAnthropic
# llm = ChatAnthropic(model="claude-3-haiku-20240307")

In [28]:
llm

ChatGoogleGenerativeAI(profile={'max_input_tokens': 1000000, 'max_output_tokens': 8192, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'image_url_inputs': True, 'pdf_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-1.5-flash', temperature=0.0, client=<google.genai.client.Client object at 0x7488911be850>, default_metadata=(), model_kwargs={})

In [ ]:
#from langchain_openai import OpenAI
#llm = OpenAI()



In [16]:
#from langchain.prompts import ChatPromptTemplate
from langchain_core.prompts import ChatPromptTemplate
template="""You are a helpful assistant. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question} 
Context: {context} 
Answer:"""
prompt = ChatPromptTemplate.from_template(template)

In [17]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [18]:
response = chain.invoke("When did Big Star Collectibles Launch?")

In [19]:
response

'Big Star Collectibles officially launched in 2014. The idea for the company was conceived in 2013 at the International Arts Conference.'